In [1]:
# 修改预训练模型的缓存目录
import os

os.environ["MODELSCOPE_CACHE"] = r"G:\code\pretrain_model_dir\_modelscope"

In [2]:
#模型下载
from modelscope import snapshot_download
model_dir = snapshot_download('BAAI/bge-reranker-base')

2025-07-05 17:00:26,288 - modelscope - INFO - Got 10 files, start to download ...


Processing 10 items:   0%|          | 0.00/10.0 [00:00<?, ?it/s]

2025-07-05 17:02:36,023 - modelscope - INFO - Download model 'BAAI/bge-reranker-base' successfully.


In [3]:
print(model_dir)

G:\code\pretrain_model_dir\_modelscope\models\BAAI\bge-reranker-base


In [11]:
from transformers import AutoTokenizer, AutoModel
import torch
# Sentences we want sentence embeddings for
queries = ['猫', '狗']
sentences = ["遇见了一只🐱", "遇见了一条🐶"]

model_dir = r"G:\code\pretrain_model_dir\_modelscope\models\BAAI\bge-reranker-base"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModel.from_pretrained(model_dir)
model.eval()

# Tokenize sentences
sentence_encoded_input = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
# for s2p(short query to long passage) retrieval task, add an instruction to query (not add instruction for passages)
instruction = "为这个句子生成表示以用于检索相关文章："
query_encoded_input = tokenizer([instruction + q for q in queries], padding=True, truncation=True, return_tensors='pt')

# Compute token embeddings
with torch.no_grad():
    sentence_model_output = model(**sentence_encoded_input)
    query_model_output = model(**query_encoded_input)
    # Perform pooling. In this case, cls pooling.
    sentence_embeddings = sentence_model_output[0][:, 0]
    query_embeddings = query_model_output[0][:, 0]
# normalize embeddings
sentence_embeddings = torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1)
query_embeddings = torch.nn.functional.normalize(query_embeddings, p=2, dim=1)
print("Sentence embeddings:", sentence_embeddings)

Some weights of XLMRobertaModel were not initialized from the model checkpoint at G:\code\pretrain_model_dir\_modelscope\models\BAAI\bge-reranker-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Sentence embeddings: tensor([[-0.0526, -0.0685, -0.0417,  ..., -0.0480, -0.0568,  0.0352],
        [-0.0564, -0.0746, -0.0369,  ..., -0.0430, -0.0523,  0.0345]])


In [6]:
sentence_embeddings.shape

torch.Size([2, 768])

In [12]:
scores = query_embeddings @ sentence_embeddings.T
print("Scores:", scores)

Scores: tensor([[0.9277, 0.8867],
        [0.8964, 0.8493]])


In [14]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_dir = r"G:\code\pretrain_model_dir\_modelscope\models\BAAI\bge-reranker-base"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
model.eval()

pairs = [['what is panda?', 'hi'], ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']]
with torch.no_grad():
    inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors='pt', max_length=512)
    scores = model(**inputs, return_dict=True).logits.view(-1, ).float()
    print(scores)

tensor([-8.1544,  6.1821])


In [3]:
from optimum.onnxruntime import ORTModelForSequenceClassification  # type: ignore

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_dir = r"G:\code\pretrain_model_dir\_modelscope\models\BAAI\bge-reranker-base"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
model_ort = ORTModelForSequenceClassification.from_pretrained(model_dir, file_name="model.onnx")

# Sentences we want sentence embeddings for
pairs = [['what is panda?', 'hi'], ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']]

# Tokenize sentences
encoded_input = tokenizer(pairs, padding=True, truncation=True, return_tensors='pt')

scores_ort = model_ort(**encoded_input, return_dict=True).logits.view(-1, ).float()
# Compute token embeddings
with torch.inference_mode():
    scores = model_ort(**encoded_input, return_dict=True).logits.view(-1, ).float()

# scores and scores_ort are identical
print(scores_ort)
print(scores)

tensor([-8.1544,  6.1821])
tensor([-8.1544,  6.1821])


In [6]:
import time
start = time.time()
scores_ort = model_ort(**encoded_input, return_dict=True).logits.view(-1, ).float()
end = time.time()
print("ORT inference time:", (end - start) * 1000, "ms")

ORT inference time: 23.00119400024414 ms


In [2]:
import onnxruntime
print(onnxruntime.get_device())
print(onnxruntime.get_available_providers())

GPU
['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
